# Week 5 – Text Classification
## Day 2: TF-IDF + Logistic Regression

**Objective:** Build and evaluate a multiclass
text classification model using TF-IDF
and Logistic Regression.

In [8]:
import pandas as pd

df = pd.read_csv("news_preprocessed.csv")

display(df.head())
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())  



,Text,Label,normalized_text
0,Budget to set scene for election\n \n Gordon B...,0,budget to set scene for election gordon brown ...
1,Army chiefs in regiments decision\n \n Militar...,0,army chiefs in regiments decision military chi...
2,Howard denies split over ID cards\n \n Michael...,0,howard denies split over id cards michael howa...
3,Observers to monitor UK election\n \n Minister...,0,observers to monitor uk election ministers wil...
4,Kilroy names election seat target\n \n Ex-chat...,0,kilroy names election seat target ex-chat show...


Dataset shape: (2127, 3)

Missing values:
Text               0
Label              0
normalized_text    0
dtype: int64


In [11]:
df = df.drop_duplicates(subset=["normalized_text"]).copy()

X = df["normalized_text"]
y = df["Label"]

print("Final dataset shape:", df.shape)

Final dataset shape: (2124, 3)


In [9]:
print("Duplicate normalized texts:",
      df.duplicated(subset=["normalized_text"]).sum())

print("Empty normalized texts:",
      df["normalized_text"].str.strip().eq("").sum())



Duplicate normalized texts: 3
Empty normalized texts: 0


In [10]:
duplicate_texts = df[df.duplicated(subset=["normalized_text"], keep=False)]

display(duplicate_texts[["normalized_text", "Label"]])

,normalized_text,Label
525,harinordoquy suffers france axe number eight i...,1
647,desailly backs blues revenge trip marcel desai...,1
669,davenport hits out at wimbledon world number o...,1
834,desailly backs blues revenge trip marcel desai...,1
841,davenport hits out at wimbledon world number o...,1
882,harinordoquy suffers france axe number eight i...,1


In [17]:
### 3. Train/Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42, 
    stratify=y 
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1699
Testing samples: 425


In [14]:
### 4. TF-IDF Feature Extraction

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (1699, 26849)
Testing TF-IDF shape: (425, 26849)


In [ ]:
### 5. Logistic Regression Model Training

from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train_tfidf, y_train)

y_pred = model.predict(X_test_tfidf)

In [16]:
### 6. Model Evaluation – Normalized Text

from sklearn.metrics import accuracy_score, f1_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9694117647058823
Macro F1: 0.9694486917783669
Weighted F1: 0.9693329324122765

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.93      0.96        81
           1       0.96      0.99      0.98       100
           2       0.97      0.97      0.97        69
           3       0.99      0.97      0.98        74
           4       0.95      0.98      0.97       101

    accuracy                           0.97       425
   macro avg       0.97      0.97      0.97       425
weighted avg       0.97      0.97      0.97       425



In [22]:
samples = df.groupby("Label").sample(n=5, random_state=42).copy()

samples["Title"] = samples["Text"].str.strip().str.split("\n").str[0]

display(samples[["Label", "Title"]].sort_values("Label"))

,Label,Title
70,0,'EU referendum could cost Â£80m'
281,0,MSPs hear renewed climate warning
283,0,Tories 'would cut number of MPs'
33,0,Blair backs 'pre-election budget'
42,0,Labour MP praises Tory campaign
811,1,Thomas out of Six Nations
618,1,Sculthorpe wants Lions captaincy
466,1,Man Utd through after Exeter test
475,1,Wasps 31-37 Leicester
433,1,O'Connell rejects Lions rumours


Label	Category	Evidence from Articles
0	Politics	   EU referendum, Tories, Labour MP
1	Sports    	Man Utd, Six Nations, Leicester
2	Technology 	Cyber crime, Firefox, PC security
3	Entertainment   	Films, music, charity shows
4	Business	    Renault, consumer spending, energy companies

In [23]:
### 3. Map Class Labels to Category Names

label_map = {
    0: "Politics",
    1: "Sports",
    2: "Technology",
    3: "Entertainment",
    4: "Business"
}

errors_df["Actual_Category"] = errors_df["Actual_Label"].map(label_map)
errors_df["Predicted_Category"] = errors_df["Predicted_Label"].map(label_map)

display(errors_df[["Actual_Category", "Predicted_Category"]])

,Actual_Category,Predicted_Category
52,Politics,Business
1818,Business,Technology
943,Technology,Entertainment
1296,Entertainment,Politics
1341,Entertainment,Sports
121,Politics,Sports
130,Politics,Business
1184,Technology,Sports
243,Politics,Business
2053,Business,Technology


In [27]:
errors_df["Original_Text"] = df.loc[errors_df.index, "Text"]

first_error = errors_df.iloc[0]

print("Actual Category:", first_error["Actual_Category"])
print("Predicted Category:", first_error["Predicted_Category"])

print("\nArticle Text:\n")
print(first_error["Original_Text"])

Actual Category: Politics
Predicted Category: Business

Article Text:

Sport betting rules in spotlight
 
 A group of MPs and peers has called for a tightening of regulations controlling betting on sport.
 
 The Parliamentary Group on Betting and Gaming held a substantial inquiry into betting last year. It followed fears that a massive increase in betting on sport, such as that done using the internet and mobile phones, has led to more cheating. The all-party group recommended 15 ways to protect punters and improve the integrity of sports betting. They include a proposal for raising the maximum jail sentence for gambling cheats above the current two years. Lord Condon, head of the International Cricket Council's anti-corruption unit, who originally made the call for longer prison sentences, said the two-year penalty was "derisory". "You could get a bigger sentence for failing to pay your hotel bill criminally than you could for corruption in major sports. "Symbolically, a higher penalt